# Oráculo: analizador de espectro de FI

Oráculo de `docs/algorithms/analizador-espectro-fi.md`, independiente de
cualquier crate Rust (todavía no existe implementación — primer paso del
método de `docs/algorithms/roadmap.md` §"Método de estudio"). Periodograma
de Welch: ventana, FFT, potencia, promediado **en potencia lineal** —la
página es explícita en que promediar en dB sesga la traza hacia abajo por
media geométrica— y conversión a dBm al final.

**La normalización correcta depende de qué se quiere leer bien, y no es la
misma para las dos partes del criterio de aceptación.** Leer correctamente
la potencia de pico de un tono coherente exige normalizar por la **ganancia
coherente** de la ventana (`Σw`); leer correctamente la densidad de un
suelo de ruido exige la **ganancia de ruido** (`Σw²`), y la diferencia entre
ambas es el ancho de banda de ruido equivalente (ENBW, Harris 1978) — que la
propia página nombra como "la ganancia de la ventana" al describir el error
de normalización más común. Los oráculos anteriores de esta serie
(`ruido-y-umbrales.md`, `gmap-clutter-filtering.md`) usaban la normalización
de ruido porque su trabajo era estimar suelo de ruido; éste usa la
coherente, porque su trabajo es mostrar un tono correctamente, y corrige la
lectura del suelo de ruido con el factor ENBW explícito en vez de mezclar
las dos normalizaciones sin decirlo.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(20260917)
plt.rcParams["figure.figsize"] = (9, 4)
np.set_printoptions(precision=4, suppress=True)

In [2]:
def complex_gaussian(rng, variance, size=None):
    sigma = np.sqrt(variance / 2.0)
    return rng.normal(0.0, sigma, size=size) + 1j * rng.normal(0.0, sigma, size=size)


def generate_tone_capture(bin_index, power_lin, m, noise_floor, rng):
    n = np.arange(m)
    phase0 = rng.uniform(-np.pi, np.pi)  # fase inicial distinta cada captura, como un tono real
    y = np.sqrt(power_lin) * np.exp(1j * (2.0 * np.pi * bin_index * n / m + phase0))
    if noise_floor > 0.0:
        y = y + complex_gaussian(rng, noise_floor, size=m)
    return y


def generate_noise_capture(noise_var, m, rng):
    return complex_gaussian(rng, noise_var, size=m)


def welch_trace_dbm(captures, win, ref_level_dbm_offset):
    """Promedia potencia LINEAL sobre las capturas (no en dB), normalización
    de ganancia coherente (Σw)² — correcta para leer el nivel de un tono."""
    sw2 = np.sum(win) ** 2
    powers = [np.abs(np.fft.fft(c * win)) ** 2 / sw2 for c in captures]
    avg_power_linear = np.mean(powers, axis=0)
    return 10.0 * np.log10(np.maximum(avg_power_linear, 1e-300)) + ref_level_dbm_offset


def enbw_bins(win):
    """Ancho de banda de ruido equivalente, en bins (Harris 1978)."""
    m = len(win)
    return m * np.sum(win ** 2) / np.sum(win) ** 2


checks = []

## Prueba 1 — posición y nivel del pico de un tono, incluido el borde

Tono de potencia digital conocida inyectado en varios bins del span,
incluidos los extremos (`0` y `M-1`, los bordes del array de FFT). El pico
de la traza debe caer en el bin correcto y su nivel, convertido a dBm con la
misma constante de referencia con la que se inyectó, debe reproducir la
potencia inyectada.

In [3]:
M = 64
WIN = np.hanning(M)
REF_LEVEL_OFFSET_DBM = -30.0  # constante de referencia ilustrativa (ganancia de receptor + calibración)
K_AVERAGES = 20
TONE_POWER_LIN = 2.0
NOISE_FLOOR_TONE_TEST = 0.001  # bajo, para que el pico del tono domine con claridad
LEVEL_TOLERANCE_DB = 0.5

expected_dbm = 10.0 * np.log10(TONE_POWER_LIN) + REF_LEVEL_OFFSET_DBM
bin_positions = [0, 1, 32, 63]
peak_ok, level_ok = True, True
print(f"nivel esperado: {expected_dbm:.2f} dBm\n{'bin':>4} {'pico_detectado':>15} {'nivel (dBm)':>12}")
for bin_idx in bin_positions:
    captures = [generate_tone_capture(bin_idx, TONE_POWER_LIN, M, NOISE_FLOOR_TONE_TEST, rng) for _ in range(K_AVERAGES)]
    trace = welch_trace_dbm(captures, WIN, REF_LEVEL_OFFSET_DBM)
    peak_bin = int(np.argmax(trace))
    print(f"{bin_idx:4d} {peak_bin:15d} {trace[peak_bin]:12.3f}")
    if peak_bin != bin_idx:
        peak_ok = False
    if abs(trace[peak_bin] - expected_dbm) >= LEVEL_TOLERANCE_DB:
        level_ok = False

checks.append(("el pico se detecta en el bin correcto en todas las posiciones probadas, incluidos los bordes", peak_ok))
checks.append((f"el nivel del pico coincide con la potencia inyectada dentro de {LEVEL_TOLERANCE_DB} dB", level_ok))

nivel esperado: -26.99 dBm
 bin  pico_detectado  nivel (dBm)
   0               0      -26.983
   1               1      -26.983
  32              32      -26.990
  63              63      -26.991


## Prueba 2 — nivel medio de la traza sobre ruido blanco: el ENBW

Con normalización de ganancia coherente, un suelo de ruido blanco **no** se
lee como su potencia real por bin — se lee escalada por el ENBW de la
ventana. Esto es exactamente "el error de normalización más común" que la
página advierte: hay que corregirlo explícitamente, no promediar más
capturas esperando que desaparezca solo (no lo hace, es un sesgo
determinista de la ventana, no ruido).

In [4]:
NOISE_VAR = 0.5
K_AVERAGES_NOISE = 2000
ENBW = enbw_bins(WIN)
print(f"ENBW de la ventana de Hann: {ENBW:.4f} bins")

noise_captures = [generate_noise_capture(NOISE_VAR, M, rng) for _ in range(K_AVERAGES_NOISE)]
noise_trace_linear = np.mean(
    [np.abs(np.fft.fft(c * WIN)) ** 2 / np.sum(WIN) ** 2 for c in noise_captures], axis=0)

measured_total = noise_trace_linear.sum()
expected_total_uncorrected = NOISE_VAR  # lo que se leeria si (erroneamente) no se corrigiera el ENBW
expected_total_corrected = NOISE_VAR * ENBW

print(f"potencia total medida en la traza: {measured_total:.4f}")
print(f"esperado SIN corregir ENBW (el error común): {expected_total_uncorrected:.4f}")
print(f"esperado corrigiendo ENBW: {expected_total_corrected:.4f}")

checks.append(("la traza reproduce ruido_var·ENBW, no ruido_var sin corregir (diferencia > 20% respecto del sin corregir)",
               abs(measured_total - expected_total_uncorrected) > 0.2 * expected_total_uncorrected))
checks.append(("corrigiendo por ENBW, la traza reproduce la densidad de ruido esperada dentro de 5%",
               abs(measured_total - expected_total_corrected) < 0.05 * expected_total_corrected))

ENBW de la ventana de Hann: 1.5238 bins
potencia total medida en la traza: 0.7636
esperado SIN corregir ENBW (el error común): 0.5000
esperado corrigiendo ENBW: 0.7619


## Prueba 3 — la dispersión de la traza baja con el número de promedios

Promediar `K` capturas en potencia (no en dB) debe reducir la varianza de
cada bin en el factor `K` — la misma ley que el promediado en rango de
`procesamiento-de-rango.md`, aquí aplicada a promedios de traza en vez de
celdas de rango contiguas.

In [5]:
K_GRID = [1, 4, 16, 64]
N_TRIALS_VAR = 1500
TEST_BIN = 20

variances = {}
for k in K_GRID:
    samples = []
    for _ in range(N_TRIALS_VAR):
        captures = [generate_noise_capture(NOISE_VAR, M, rng) for _ in range(k)]
        powers = [np.abs(np.fft.fft(c * WIN)) ** 2 / np.sum(WIN) ** 2 for c in captures]
        samples.append(np.mean(powers, axis=0)[TEST_BIN])
    variances[k] = np.var(samples, ddof=1)
    print(f"K={k:3d}  var(bin {TEST_BIN})={variances[k]:.6f}  "
          f"razón var(K=1)/var(K)={variances[1]/variances[k]:.2f}  esperado={k}")

checks.append(("la varianza de la traza decrece en el factor K esperado (dentro de 15%)",
               all(abs(variances[1] / variances[k] - k) < 0.15 * k for k in K_GRID[1:])))

K=  1  var(bin 20)=0.000148  razón var(K=1)/var(K)=1.00  esperado=1


K=  4  var(bin 20)=0.000037  razón var(K=1)/var(K)=3.96  esperado=4


K= 16  var(bin 20)=0.000008  razón var(K=1)/var(K)=17.95  esperado=16


K= 64  var(bin 20)=0.000002  razón var(K=1)/var(K)=66.57  esperado=64


## Fuera de alcance, declarado

- **Selección de canal, span y frecuencia central desde la sintonía del
  NCO**: mapeo de configuración (`center_freq_hz`, `span_hz` a partir de la
  frecuencia de muestreo del DRx), no un algoritmo con verdad-terreno
  numérica propia más allá de la posición de bin ya cubierta en la Prueba 1.
- **Captura oportunista sobre el flujo vivo vs. modo dedicado**: decisión de
  arquitectura de dónde se toman las muestras, no una cantidad que este
  oráculo pueda ejercitar en simulación.

In [6]:
all_ok = True
for name, ok in checks:
    print(f"[{'OK' if ok else 'FALLO'}] {name}")
    all_ok = all_ok and ok

assert all_ok, "el oráculo del analizador de espectro de FI no pasa todas las comprobaciones — ver tabla arriba"
print("\nTodas las comprobaciones dentro de tolerancia.")

[OK] el pico se detecta en el bin correcto en todas las posiciones probadas, incluidos los bordes
[OK] el nivel del pico coincide con la potencia inyectada dentro de 0.5 dB
[OK] la traza reproduce ruido_var·ENBW, no ruido_var sin corregir (diferencia > 20% respecto del sin corregir)
[OK] corrigiendo por ENBW, la traza reproduce la densidad de ruido esperada dentro de 5%
[OK] la varianza de la traza decrece en el factor K esperado (dentro de 15%)

Todas las comprobaciones dentro de tolerancia.
